# Create Lakehouse from Unity Catalog

This notebook creates schemas and tables in the current default notebooks default lakehouse that point to the underlinyg physical location used by Unity Catalog.

## Requirements:
- A Fabric Connection of type `Azure Databricks Workspace`
- This notebook with the Fabric Connection from above added to its notebook connections
- The internal ID of that notebook connection
- The name of the Unity Catalog catalog that you want to link into the notebooks default lakehouse


## Parameters

In [12]:
# connection *within* notebook - must be of type "Azure Databricks Workspace"
# the Fabric connection must be added to the notebook first in order to access and use it
notebook_connection_id = "90f19b0e-71ca-4518-98d0-4ed5ab2597b4" 

# name of the Unity Catalog catalog that you want to link
uc_catalog_name = "uc_on_lakehouse"
drop_table_if_exists = True

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 14, Finished, Available, Finished, False)

## Setup
- install the [Databricks SDK](https://databricks-sdk-py.readthedocs.io/en/stable/getting-started.html)

In [13]:
!pip install databricks-sdk

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 15, Finished, Available, Finished, False)

In [1]:
import json
import time
import requests

import sempy.fabric as fabric
from databricks.sdk import WorkspaceClient

StatementMeta(, c6d62b04-f71c-4aa7-b61a-2cff40ebc1e9, 3, Finished, Available, Finished, False)

## Get all relevant information from the Notebook Connection
- OAuth token
- Actual Connection ID within Fabric

In [15]:
connection_credential = notebookutils.connections.getCredential(notebook_connection_id)
credential_dict = json.loads(connection_credential['credential'])

databricks_token = [x["value"] for x in credential_dict["credentialData"] if x["name"] == "AccessToken"][0]
connection_id_fabric = connection_credential["datasourceId"] # actual Fabric connection ID

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 17, Finished, Available, Finished, False)

## Get all relevant information from the underlying Fabric Connection
- hostname

In [16]:
token = notebookutils.credentials.getToken("pbi")

headers = {
    "Authorization": f"Bearer {token}"
}

resp = requests.get(f"https://api.fabric.microsoft.com/v1/connections/{connection_id_fabric}", headers = headers)

conn_details = resp.json()

hostname = conn_details["connectionDetails"]["path"]

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 18, Finished, Available, Finished, False)

## Seutp Databricks Workspace Client using the OAuth token and hostname

In [17]:
w = WorkspaceClient(
    host = hostname,
    token = databricks_token,
    auth_type = "pat",  # its not actually a PAT but our OAuth token. However, its just ued as-is and works
)

me = w.current_user.me()
print(me.user_name)

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 19, Finished, Available, Finished, False)

5b312334-1371-4fe4-ae1e-2260ffc4a956


## Get the Storage root Path from the Unity Catalog that we want to link
> The catalog not necessary has to be stored on OneLake, but it must be accessible from Fabric!

In [18]:
catalog = w.catalogs.get(uc_catalog_name)
storage_root_path = catalog.storage_location

print(f"{storage_root_path = }")

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 20, Finished, Available, Finished, False)

storage_root_path = 'abfss://f5474d5d-5ab8-4207-96ac-0c747f57b3da@onelake.dfs.fabric.microsoft.com/4afe04c6-11fc-47f1-b60e-c6ca6ed18a1e/Files/UnityCatalog/__unitystorage/catalogs/f6345787-23a2-477f-9699-6ced2b8ff5e4'


## Get all Schemas and Tables from the Unity Catalog
- schema name and IDs
- table name and IDs and containing schema

In [19]:
uc_schemas = [s for s in w.schemas.list(catalog_name=catalog.name) if s.name != "information_schema"]

schemas = {}
all_tables = []
for uc_schema in uc_schemas:
    schemas[uc_schema.name] = uc_schema.schema_id
    tables = w.tables.list(catalog_name=catalog.name, schema_name=uc_schema.name)
    
    all_tables.extend([t for t in tables])

[(t.schema_name, t.name, t.table_id) for t in all_tables]

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 21, Finished, Available, Finished, False)

[('default', 'my_other_table', '2df3de0d-154c-42a6-9409-2b7170d274b1'),
 ('default', 'my_table', '393d776e-e398-4e99-8dfb-386f225365b9'),
 ('my_schema', 'my_other_table', 'c566221d-e6e1-47d6-b602-66d5805e7d97')]

## Create schemas in the Lakehouse

In [20]:
for schema in schemas:
    sql_cmd = f"CREATE SCHEMA IF NOT EXISTS {schema};"
    print(sql_cmd)
    spark.sql(sql_cmd)

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 22, Finished, Available, Finished, False)

CREATE SCHEMA IF NOT EXISTS default;
CREATE SCHEMA IF NOT EXISTS my_schema;


## Create all tables in the Lakehouse

In [26]:
for table in all_tables:
    table_name = f"{table.schema_name}.{table.name}"
    
    if drop_table_if_exists:
        sql_cmd = f"DROP TABLE IF EXISTS {table_name}"
        print(sql_cmd)
        spark.sql(sql_cmd)

# it can take a bit for the tables to be properly dropped  
# and metadata is updated 
time.sleep(10)

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 28, Finished, Available, Finished, False)

DROP TABLE IF EXISTS default.my_other_table
DROP TABLE IF EXISTS default.my_table
DROP TABLE IF EXISTS my_schema.my_other_table


In [27]:
for table in all_tables:
    table_path = f"{storage_root_path}/tables/{table.table_id}"
    table_name = f"{table.schema_name}.{table.name}"
    
    sql_cmd = f"CREATE EXTERNAL TABLE {table_name} \nUSING DELTA\nLOCATION '{table_path}'"
    print(sql_cmd)
    spark.sql(sql_cmd)

StatementMeta(, c8b6fce6-e22d-4818-99e4-46161fbfa3c8, 29, Finished, Available, Finished, False)

DROP TABLE IF EXISTS default.my_other_table
CREATE EXTERNAL TABLE default.my_other_table 
USING DELTA
LOCATION 'abfss://f5474d5d-5ab8-4207-96ac-0c747f57b3da@onelake.dfs.fabric.microsoft.com/4afe04c6-11fc-47f1-b60e-c6ca6ed18a1e/Files/UnityCatalog/__unitystorage/catalogs/f6345787-23a2-477f-9699-6ced2b8ff5e4/tables/2df3de0d-154c-42a6-9409-2b7170d274b1'
DROP TABLE IF EXISTS default.my_table
CREATE EXTERNAL TABLE default.my_table 
USING DELTA
LOCATION 'abfss://f5474d5d-5ab8-4207-96ac-0c747f57b3da@onelake.dfs.fabric.microsoft.com/4afe04c6-11fc-47f1-b60e-c6ca6ed18a1e/Files/UnityCatalog/__unitystorage/catalogs/f6345787-23a2-477f-9699-6ced2b8ff5e4/tables/393d776e-e398-4e99-8dfb-386f225365b9'
DROP TABLE IF EXISTS my_schema.my_other_table
CREATE EXTERNAL TABLE my_schema.my_other_table 
USING DELTA
LOCATION 'abfss://f5474d5d-5ab8-4207-96ac-0c747f57b3da@onelake.dfs.fabric.microsoft.com/4afe04c6-11fc-47f1-b60e-c6ca6ed18a1e/Files/UnityCatalog/__unitystorage/catalogs/f6345787-23a2-477f-9699-6ced2b8ff5